# HumorVibes — Corpus Lab: scan, measure, remix the internet's jokes

Fetch jokes from free public APIs, measure each with the local Gemma instrument (S surprise / R resolution net of a decoy-null / E efficiency, frame guessed few-shot with an honest NONE), map the whole corpus onto the laugh region, then **remix the best frames into other formats** and measure whether the frame survives the transfer.

*(Licensing note: text jokes from APIs built to serve jokes; no performer clips are scraped — clip work belongs to licensed corpora or original material rendered via ClipPlan.)*

In [ ]:
import glob, json, os, re, time, urllib.request, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
os.makedirs('/kaggle/working/research_out', exist_ok=True)
UA = {'User-Agent': 'HumorVibes research notebook (Kaggle; Humor Genome NYC hackathon)'}

def fetch_json(url, headers=None, timeout=20):
    req = urllib.request.Request(url, headers={**UA, **(headers or {})})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.loads(r.read().decode('utf-8'))

corpus = []
try:
    for _ in range(15):
        d = fetch_json('https://icanhazdadjoke.com/', headers={'Accept': 'application/json'})
        if d.get('joke'): corpus.append({'src': 'icanhazdadjoke', 'text': d['joke']})
        time.sleep(0.6)
except Exception as e:
    print('dadjoke fetch stopped:', e)
try:
    d = fetch_json('https://v2.jokeapi.dev/joke/Any?safe-mode&type=twopart&amount=10')
    for j in d.get('jokes', []):
        corpus.append({'src': 'jokeapi', 'text': j['setup'] + ' ' + j['delivery']})
    d = fetch_json('https://v2.jokeapi.dev/joke/Any?safe-mode&type=single&amount=10')
    for j in d.get('jokes', []):
        corpus.append({'src': 'jokeapi', 'text': j['joke']})
except Exception as e:
    print('jokeapi fetch stopped:', e)
seen, deduped = set(), []
for item in corpus:
    key = item['text'][:60].lower()
    if key not in seen:
        seen.add(key); deduped.append(item)
corpus = deduped
print(f'corpus: {len(corpus)} unique jokes')
for c in corpus[:5]: print(' -', c['text'][:80])

In [ ]:
gcfg = [p for p in glob.glob('/kaggle/input/**/config.json', recursive=True) if 'gemma' in p.lower()]
MODEL_PATH = os.path.dirname(gcfg[0])
tok = AutoTokenizer.from_pretrained(MODEL_PATH)
def load_fb(path):
    if torch.cuda.is_available():
        try:
            m = AutoModelForCausalLM.from_pretrained(path, torch_dtype=torch.float16, device_map='auto').eval()
            with torch.no_grad(): m(torch.tensor([[tok.bos_token_id or 2]]).to(m.device))
            return m
        except Exception as e:
            print('cuda->cpu:', str(e)[:90]); torch.cuda.empty_cache()
    return AutoModelForCausalLM.from_pretrained(path, torch_dtype=torch.float32).eval()
model = load_fb(MODEL_PATH)
print('instrument:', model.device)

def nll_mean(context, continuation):
    ctx = tok(context, return_tensors='pt').input_ids
    cont = tok(continuation, add_special_tokens=False, return_tensors='pt').input_ids
    full = torch.cat([ctx, cont], dim=1).to(model.device)
    with torch.no_grad():
        lp = torch.log_softmax(model(full).logits.float(), dim=-1)
    n = ctx.shape[1]
    vals = [float(-lp[0, n+i-1, int(full[0, n+i])]) for i in range(cont.shape[1])]
    return sum(vals) / len(vals)

def gen(prompt, max_new=70, temperature=0.4):
    ids = tok.apply_chat_template([{'role':'user','content':prompt}], return_tensors='pt', add_generation_prompt=True)
    if not torch.is_tensor(ids): ids = ids['input_ids']
    ids = ids.to(model.device)
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=max_new, do_sample=True, temperature=temperature,
                             top_p=0.95, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True).strip()

FEWSHOT = ("A joke works because a hidden frame reinterprets the punchline - the fact that, once stated, "
           "makes the punchline the OBVIOUS next thing to say.\n"
           "Example - Setup: I told my therapist about my fear of speed bumps. "
           "Punchline: She said I'm slowly getting over it. "
           "Frame: 'Getting over it' is literal - the car physically drives over the bumps slowly.\n")
DECOY = 'It turns out this is really about quarterly regional cheese sales figures.'
S_LO, S_HI = 1.2, 5.5

def split_sp(text):
    for sep in ['\n', '. ', '? ', '! ', ' - ', ': ']:
        if sep in text:
            a, b = text.rsplit(sep, 1)
            if len(b.split()) >= 2: return a + sep.strip(), b.strip()
    w = text.split(); c = max(1, int(len(w)*0.7))
    return ' '.join(w[:c]), ' '.join(w[c:])

def measure(text):
    setup, punch = split_sp(text)
    S = nll_mean(setup + '\n', ' ' + punch)
    frame = gen(FEWSHOT + 'Now - Joke: ' + text + '\nFrame (ONE short sentence, no preamble; if none, NONE):',
                max_new=50, temperature=0.3).splitlines()[0].strip()
    if not frame or frame.upper().startswith('NONE'):
        return dict(S=round(S,3), R=0.0, E=0.0, frame='NONE', setup=setup, punch=punch)
    r_raw = max(0.0, S - nll_mean(setup + '\n(' + frame + ')\n', ' ' + punch))
    r_null = max(0.0, S - nll_mean(setup + '\n(' + DECOY + ')\n', ' ' + punch))
    R = max(0.0, r_raw - r_null)
    return dict(S=round(S,3), R=round(R,3), E=round(R/max(1,len(frame.split())),4),
                frame=frame[:100], setup=setup, punch=punch)

## Measure the corpus: where does the internet's humor sit in the laugh region?

In [ ]:
MAXN = 30 if model.device.type == 'cpu' else 60
measured = []
t0 = time.time()
for i, item in enumerate(corpus[:MAXN]):
    try:
        m = measure(item['text'])
    except Exception as e:
        print('skip', i, str(e)[:60]); continue
    m['src'] = item['src']; m['text'] = item['text']
    measured.append(m)
    if (i+1) % 5 == 0: print(f'{i+1}/{min(len(corpus), MAXN)} measured ({time.time()-t0:.0f}s)')
in_band = [m for m in measured if S_LO < m['S'] < S_HI and m['R'] >= 0.3]
predictable = [m for m in measured if m['S'] <= S_LO]
no_frame = [m for m in measured if m['frame'] == 'NONE' or m['R'] < 0.05]
print(f'\ncensus: {len(measured)} measured | laugh-region {len(in_band)} | '
      f'predictable {len(predictable)} | weak/no frame {len(no_frame)}')
ranked = sorted(measured, key=lambda m: -(m['R'] + 0.2*min(m['S'], S_HI)))
print('\nTOP 5 (measured):')
for m in ranked[:5]: print(f"  S={m['S']:5.2f} R={m['R']:5.2f} :: {m['text'][:70]}")
print('\nBOTTOM 3:')
for m in ranked[-3:]: print(f"  S={m['S']:5.2f} R={m['R']:5.2f} :: {m['text'][:70]}")

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 5))
ax.axvspan(S_LO, S_HI, alpha=0.08, color='green')
for m in measured:
    ax.scatter(m['S'], m['R'], s=60, alpha=0.7,
               c='#2a9d3a' if (S_LO < m['S'] < S_HI and m['R'] >= 0.3) else '#5b8dad')
ax.set_xlabel('S - surprise (nats)'); ax.set_ylabel('R - resolution (net of null)')
ax.set_title(f'The internet corpus on the laugh region (n={len(measured)})\ngreen = measured laugh-region items')
plt.tight_layout(); plt.show()

## Temporal: does the joke rent a hot cache or the population's deep cache?
THEORY.md §11: state the fact as an explicit frame hint (R_with) vs strip it and see how much the model's OWN cache already explains the punchline (R_without = NLL('Someone says:', punchline) minus NLL(setup, punchline)). Small gap -> the population cache already carries the joke (canonical, evergreen). Large gap -> the joke only resolves once the fact is stated - it rents a hot, shallow cache (topical) that evicts as the news cycle moves on.

In [ ]:
FEED = 'https://feeds.bbci.co.uk/news/technology/rss.xml'
headlines = []
try:
    req = urllib.request.Request(FEED, headers=UA)
    with urllib.request.urlopen(req, timeout=20) as r:
        xml = r.read().decode('utf-8', 'replace')
    titles = re.findall(r'<item>.*?<title>(?:<!\[CDATA\[)?(.*?)(?:\]\]>)?</title>', xml, flags=re.DOTALL)
    headlines = [' '.join(t.split()) for t in titles if len(' '.join(t.split())) >= 15][:8]
except Exception as e:
    print('rss fetch stopped:', e)
print(f'fetched {len(headlines)} headlines')

FACTS = [
    ('Icarus', 'Icarus flew too close to the sun on wings of wax and feathers, and they melted.'),
    ('Trojan horse', 'The Greeks hid soldiers inside a giant wooden horse to sneak into Troy.'),
    ("Newton's apple", 'An apple falling from a tree helped Isaac Newton work out that gravity pulls things down.'),
    ('Eureka', 'Archimedes shouted Eureka and ran from his bath naked after realizing displaced water reveals volume.'),
]

JOKE_ASK = ('Write ONE short one-liner joke (a setup then a punchline, no more than two sentences) '
            'that only lands if you know this fact: {fact}\nReturn only the joke, no preamble, no quotes.')

def temporal_gap(fact, one_liner):
    setup, punch = split_sp(one_liner)
    S = nll_mean(setup + '\n', ' ' + punch)
    r_raw = max(0.0, S - nll_mean(setup + '\n(' + fact + ')\n', ' ' + punch))
    r_null = max(0.0, S - nll_mean(setup + '\n(' + DECOY + ')\n', ' ' + punch))
    R_with = max(0.0, r_raw - r_null)
    bare = nll_mean('Someone says:\n', ' ' + punch)
    R_without = max(0.0, round(bare - S, 3))
    gap = max(0.0, round(R_with - R_without, 3))
    if gap < 0.3: verdict = 'canonical'
    elif gap >= 0.8: verdict = 'topical-cache'
    else: verdict = 'mixed'
    return dict(setup=setup, punch=punch, R_with=round(R_with, 3), R_without=R_without, gap=gap, verdict=verdict)

temporal_results = []
for headline in headlines[:4]:
    one_liner = gen(JOKE_ASK.format(fact=headline), max_new=70, temperature=0.7)
    row = temporal_gap(headline, one_liner)
    row.update(kind='topical', label=headline[:70], fact=headline, joke=one_liner[:160])
    temporal_results.append(row)
for label, fact in FACTS:
    one_liner = gen(JOKE_ASK.format(fact=fact), max_new=70, temperature=0.7)
    row = temporal_gap(fact, one_liner)
    row.update(kind='canonical', label=label, fact=fact, joke=one_liner[:160])
    temporal_results.append(row)

print(f"\n{'kind':10s} {'label':22s} {'gap':>5s}  verdict")
for row in temporal_results:
    print(f"{row['kind']:10s} {row['label'][:22]:22s} {row['gap']:5.2f}  {row['verdict']:14s} :: {row['joke'][:60]}")

## Remix: does the frame survive a format transfer?
Take the top-measured frames and recompile them into a different timing envelope (meme caption / 15-second beat sheet). THEORY.md says the *frame* is the invariant and the *format* is the envelope — so a good remix keeps R when the surface changes.

In [ ]:
REMIX_FORMATS = {
  'meme_caption': "Rewrite as a meme: 'TOP: ... / BOTTOM: ...', <=10 words each; bottom re-frames, never describes.",
  'shorts_beat': "Rewrite as 'HOOK:/BUILD:/SNAP:' beats, <=40 spoken words total.",
}
remixes = []
for m in ranked[:3]:
    if m['frame'] == 'NONE': continue
    for fmt, contract in REMIX_FORMATS.items():
        out = gen(f"Keep this exact comic frame: {m['frame']}\nOriginal joke: {m['text']}\n"
                  f"{contract} Return only the rewritten piece.", max_new=90, temperature=0.8)
        s2, p2 = split_sp(out.replace('/', '\n'))
        S2 = nll_mean(s2 + '\n', ' ' + p2)
        r_raw2 = max(0.0, S2 - nll_mean(s2 + '\n(' + m['frame'] + ')\n', ' ' + p2))
        r_null2 = max(0.0, S2 - nll_mean(s2 + '\n(' + DECOY + ')\n', ' ' + p2))
        R2 = max(0.0, r_raw2 - r_null2)
        kept = 'FRAME SURVIVED' if R2 >= 0.5 * max(m['R'], 0.05) else 'frame lost'
        remixes.append({'orig': m['text'][:70], 'frame': m['frame'], 'format': fmt,
                        'remix': out[:160], 'R_orig': m['R'], 'R_remix': round(R2,3), 'verdict': kept})
        print(f"[{fmt}] R {m['R']:.2f} -> {R2:.2f} ({kept})\n  {out[:110]}\n")
json.dump({'measured': measured, 'remixes': remixes, 'temporal': temporal_results},
          open('/kaggle/working/research_out/corpus_report.json', 'w'), indent=2)
print('wrote corpus_report.json')

## Reading the results
- The census is a measured claim about internet humor: what fraction actually sits in the laugh region vs being predictable (dad-joke floor) or frame-less.
- Remix verdicts test the theory's central invariance: **frames transfer, surfaces don't** — a remix that keeps ≥50% of the original's R carried its re-route into the new envelope.
- With hosted keys attached (Gemini add-on / Ollama Cloud / NVIDIA / Mistral), the same pipeline upgrades: better frame-writers, persona-panel ratings per item, and vibe-matched remixing per target audience.